#### **<span style="color: maroon;">Natural Language Processing with Disaster Tweets</span>**

Researcher: [Andrew Batyrov](https://www.kaggle.com/semipro), (C) 2025

##### Table of Contents

1. [Description](#description)

2. [Load Data and basic EDA](#load-data)

3. [DeBERTa fine-tuning](#deberta)

4. [Make predictions](#predict)

---
##### <a id="description"></a>1. Description

https://www.kaggle.com/competitions/nlp-getting-started/

In this competition, you’re challenged to build a machine learning model that predicts which Tweets are about real disasters and which one’s aren’t. You’ll have access to a dataset of 10,000 tweets that were hand classified. If this is your first time working on an NLP problem, we've created a quick tutorial to get you up and running.

Disclaimer: The dataset for this competition contains text that may be considered profane, vulgar, or offensive.

Submissions are evaluated using F1 between the predicted and expected answers.

In [ ]:
# Essential import
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import torch
#from transformers import DebertaForSequenceClassification, DebertaTokenizer
from transformers import DebertaV2ForSequenceClassification, DebertaV2Tokenizer
from tqdm import tqdm
import matplotlib.pyplot as plt
seed = 500
torch.manual_seed(seed=seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'{device = }')

In [ ]:
# Clean-up if needed
!rm -rf /kaggle/working/*

In [ ]:
def save_metrics(model_name, **metrics):
    '''
    Dump train/val loss and f1-score lists to a json file.
    '''
    with open(f'./metrics/{model_name}.json', 'w') as fh:
        json.dump(metrics, fh)

In [ ]:
def load_metrics(model_name):
    '''
    Load train/val loss and f1-score lists from a json file.
    '''
    try:
        with open(f'./metrics/{model_name}.json', 'r') as fh:
            return json.load(fh)
    except:
        raise RuntimeError(f'The metrics json file not found. Was the model `{model_name}` trained?')

In [ ]:
def load_model(model):
    '''
    Try to find and load our fine-tuned model.
    '''
    try:
        model.model.load_state_dict(torch.load(f'./models/{model.name}.pt'))
        model.model = model.model.to(device)
        model.is_trained = True
        print(f'Found fine-tuned model `{model.name}.pt`. Ready for inferring/testing.')
    except:
        print(f'Fine-tuned model `{model.name}` not found. Train/eval the model first.')

In [ ]:
def plot_metrics(model_name: str, epochs):
    '''
    Plot model's loss and f1-score metrics obtained during train/val over the number of `epochs`.
    '''
    metrics = load_metrics(model_name)
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(range(1, epochs+1), metrics['train_losses'], marker='s', markersize=5, color='C0', label='Train')
    plt.plot(range(1, epochs+1), metrics['val_losses'], marker='s', markersize=5, color='C1', label='Val')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.xticks(range(1, epochs+1))
    plt.legend()
    plt.grid(lw=0.25, color='xkcd:cement')
    plt.subplot(1, 2, 2)
    plt.plot(range(1, epochs+1), metrics['train_f1s'], marker='s', markersize=5, color='C0', label='Train')
    plt.plot(range(1, epochs+1), metrics['val_f1s'], marker='s', markersize=5 , color='C1', label='Val')
    plt.xlabel('Epoch')
    plt.ylabel('F1-score')
    plt.xticks(range(1, epochs+1))
    plt.legend()
    plt.grid(lw=0.25, color='xkcd:cement')
    plt.suptitle(f'Train and validation datasets metrics: {model_name}', fontsize=11)
    plt.tight_layout();

##### <a id="load-data"></a>2. Load Data and basic EDA

First, let's load the `train.csv` and `test.csv` data and look into it.

In [ ]:
# Load data
train = pd.read_csv('../input/nlp-getting-started/train.csv', dtype={'target': np.int64})
test = pd.read_csv('../input/nlp-getting-started/test.csv', dtype={'target': np.int64})

In [ ]:
train.shape, test.shape

In [ ]:
# Look at 10 random tweets in train
train.sample(10)

In [ ]:
# Examine target
target = 'target'

In [ ]:
# Check nulls
train[target].isna().sum()

In [ ]:
# Target distribution in train
train[target].value_counts(normalize=True)

In [ ]:
plt.figure(figsize=(3, 3))
plt.bar(train[target].value_counts().index, train[target].value_counts(), width=0.25, alpha=0.5, edgecolor='C0', color='white', hatch='.'*6)
plt.xticks(train[target].value_counts().index)
plt.title('train', size=10, fontfamily='monospace')
plt.grid(axis='y', lw=0.5, color='xkcd:cement')
plt.gca().set_axisbelow(True);

We can see that the target classes are imbalanced, though not severely.

In [ ]:
# Train text length distribution in words
train['text'].str.split().apply(len).describe()

In [ ]:
# Test text length distribution in words
test['text'].str.split().apply(len).describe()

In [ ]:
plt.figure(figsize=(4, 4))
plt.hist(train['text'].str.split().apply(len), bins=15, density=True, color='C0', alpha=0.75, histtype='step', hatch='.'*5, label='Train')
plt.hist(test['text'].str.split().apply(len), bins=15, density=True, color='C2', alpha=0.75, histtype='step', hatch='.'*5, label='Test')
plt.title('Text length, words', size=10, fontfamily='monospace')
plt.legend();

We can see that texts are mostly under 30 words long with mean around 15 words long.

##### <a id="deberta"></a>3. DeBERTa fine-tuning

We will fine-tune a pre-trained DeBERTa NLP model for our classification task. Let's choose the `deberta-base` model.

In [ ]:
# Drop duplicates in the train, if any
train = train.drop_duplicates(subset=['keyword', 'location', 'text', 'target'])

In [ ]:
# Stratified split, since target classes are imbalanced
train, val = train_test_split(train, test_size=0.05, stratify=train[target], random_state=seed)

In [ ]:
train[target].value_counts(normalize=True)

In [ ]:
val[target].value_counts(normalize=True)

First, let's write a function to prepare our data. We'll use DebertaTokenizer from Transformers for text tokenization.

In [ ]:
def prepare_data(model, dataset):
    '''
    Tokenize all sentences in `dataset` Pandas DataFrame and prepare data for DataLoader.
    The first column must me sentences, and the second column must be labels.
    Tokenizer is used from `model`.
    Source: https://mccormickml.com/2019/07/22/BERT-fine-tuning/
    '''
    sentences = dataset.iloc[:, 0].values
    labels = dataset.iloc[:, 1].values
    input_ids = []
    attention_masks = []
    for sent in sentences:
        encoded_dict = model.tokenizer.encode_plus(
                            sent,
                            add_special_tokens=True, # Add '[CLS]' and '[SEP]' special tokens
                            max_length=model.max_len,
                            padding='max_length',
                            truncation=True,
                            return_attention_mask=True,
                            return_tensors='pt'
                    )
        input_ids.append(encoded_dict['input_ids'])
        attention_masks.append(encoded_dict['attention_mask'])
    
    input_ids = torch.cat(input_ids, dim=0)
    attention_masks = torch.cat(attention_masks, dim=0)
    labels = torch.tensor(labels)
    prepared = torch.utils.data.TensorDataset(input_ids, attention_masks, labels)
    
    return prepared

Next, let's concatenate keyword and location with the main tweet's text.

In [ ]:
train['doc'] = train['text'] + ' keyword_' + train['keyword'].astype(str) + ' location_' + train['location'].astype(str).apply(lambda x: ' location_'.join(x.split()))
val['doc'] = val['text'] + ' keyword_' + val['keyword'].astype(str) + ' location_' + val['location'].astype(str).apply(lambda x: ' location_'.join(x.split()))
test['doc'] = test['text'] + ' keyword_' + test['keyword'].astype(str) + ' location_' + test['location'].astype(str).apply(lambda x: ' location_'.join(x.split()))

In [ ]:
train['doc'].iloc[100]

In [ ]:
# Check train text length distribution in words again
train['doc'].str.split().apply(len).describe()

In [ ]:
# Check test text length distribution in words again
test['doc'].str.split().apply(len).describe()

Now, let's create a class (model factory) to build models based on some pre-trained ones, but with the untrained last classification layer. The class will also have methods to train and validate models.

> Note: We can see that the max length of a text appended with keyword and location is 36 tokens. Let's set the `max_length` parameter of DeBERTa tokenizer to 64 (nearest power of 2).

In [ ]:
class TweetClassifier:

    def __init__(self, base_model: str, max_len=64, verbose=False):
        # 1. Create an existing pre-trained model
        if base_model not in ('deberta'):
            raise ValueError("base_model should be 'deberta'")
        # https://huggingface.co/microsoft/deberta-v3-base
        name = 'deberta'
        model_str = 'microsoft/deberta-v3-base'
        tokenizer_str = 'microsoft/deberta-v3-base'
        
        self.name = name
        self.model = DebertaV2ForSequenceClassification.from_pretrained(model_str, 
                                                                      num_labels=2, 
                                                                      output_attentions=False, 
                                                                      output_hidden_states=False)
        self.tokenizer = DebertaV2Tokenizer.from_pretrained(tokenizer_str)
        self.max_len = max_len
        self.is_trained = False
        self.verbose = verbose
        self.model = self.model.to(device)


    def train(self, train_subset, val_subset, optimizer, epochs: int, batch_size: int):
        # 1. Prepare data
        prepared_train = prepare_data(self, train_subset)
        prepared_val = prepare_data(self, val_subset)
        # 2. Create data loaders
        self.train_loader = torch.utils.data.DataLoader(prepared_train, batch_size=batch_size, shuffle=True, num_workers=4)
        self.val_loader = torch.utils.data.DataLoader(prepared_val, batch_size=batch_size, shuffle=True, num_workers=4)
        # 3. Store optimizer
        self.optimizer = optimizer
        # 4. Do train and eval loop for the number of epochs
        self.train_losses, self.val_losses, self.train_f1s, self.val_f1s = [], [], [], []
        for epoch_i, _ in enumerate(range(epochs), start=1):
            print(f'-------------\nEpoch: {epoch_i:>2}/{epochs}')
            # Train one epoch
            self._train()
            # Eval one epoch
            self._validate()
    

    def _train(self):
        '''
        Train the classifier on all batches of the `train_loader` for one epoch.
        '''
        # 1. Switch to train mode
        self.model.train()
        # 2. Do train loop for all batches for one epoch
        print('\nTraining...')
        losses, f1s = [], []
        n_batches_to_show = np.ceil(len(self.train_loader) / 10)
        for i, data in tqdm(enumerate(self.train_loader), total=len(self.train_loader)):
            # `data` has 3 tensors, as returned by `prepare_data()`
            input_ids = data[0].to(device)
            input_mask = data[1].to(device)
            labels = data[2].to(device)
            self.optimizer.zero_grad()
            output = self.model(input_ids, token_type_ids=None, attention_mask=input_mask, labels=labels)
            loss = output.loss
            logits = output.logits
            loss.backward()
            self.optimizer.step()
            losses.append(loss.item())
            f1 = f1_score(labels.cpu().numpy(), logits.argmax(-1).cpu().numpy())
            f1s.append(f1)
            if self.verbose:
                if i % n_batches_to_show == 0:
                    print(f'Batch: {i:>4}, Train loss: {loss:.5f}, Train f1-score: {f1:.5f}')
        # 3. Keep and report train metrics for one epoch
        epoch_mean_loss = np.mean(losses)
        self.train_losses.append(epoch_mean_loss)
        epoch_mean_f1 = np.mean(f1s)
        self.train_f1s.append(np.mean(epoch_mean_f1))
        print(f'Epoch avg: Train loss: {epoch_mean_loss:.5f}, Train f1-score {epoch_mean_f1:.5f}')


    def _validate(self):
        '''
        Validate the classifier on all batches of the `val_loader` for one epoch.
        '''
        # 1. Switch to eval mode
        self.model.eval()
        # 2. Do eval loop
        print('\nEvaluating...')
        losses, f1s = [], []
        n_batches_to_show = np.ceil(len(self.val_loader) / 10)
        with torch.no_grad():
            for i, data in tqdm(enumerate(self.val_loader), total=len(self.val_loader)):
                # `data` has 3 tensors, as returned by `prepare_data()`
                input_ids = data[0].to(device)
                input_mask = data[1].to(device)
                labels = data[2].to(device)
                output = self.model(input_ids, token_type_ids=None, attention_mask=input_mask, labels=labels)
                loss = output.loss
                logits = output.logits
                losses.append(loss.item())
                f1 = f1_score(labels.cpu().numpy(), logits.argmax(-1).cpu().numpy())
                f1s.append(f1)
                if self.verbose:
                    if i % n_batches_to_show == 0:
                        print(f'Batch: {i:>3}, Val loss: {loss:.5f}, Val f1-score: {f1:.5f}')
        # 3. Keep and report val metrics for one epoch
        epoch_mean_loss = np.mean(losses)
        self.val_losses.append(epoch_mean_loss)
        epoch_mean_f1 = np.mean(f1s)
        self.val_f1s.append(np.mean(epoch_mean_f1))
        print(f'Epoch avg: Val loss: {epoch_mean_loss:.5f}, Val f1-score: {epoch_mean_f1:.5f}')

Now, let's write a function to set up training parameters and invoke model training. Let's choose Adam optimizer with learning rate ~ 2e-6 and train for 6 epochs with batch size = 64. The learning rate may look too small, but this is done intentionally to decrease overfitting, since DeBERTa is already trained in fact, and we're just fine-tuning it.

In [ ]:
def train_model(model, train_subset, val_subset, epochs, batch_size):
    ''' 
    Set up training parameters and invoke model training.
    '''
    if not model.is_trained:
        print(f'Training model `{model.name}`.')
        lr = 1.9e-6 # 1e-5
        optimizer = torch.optim.Adam(model.model.parameters(), lr=lr)
        model.train(train_subset, val_subset, optimizer, epochs, batch_size)
        model.is_trained = True
        # Save model's state and train/val metrics to disk
        torch.save(model.model.state_dict(), f'./models/{model.name}.pt')
        save_metrics(model.name, train_losses=model.train_losses, val_losses=model.val_losses, train_f1s=model.train_f1s, val_f1s=model.val_f1s)
        print('Model successfully trained and saved to disk.')
    else:
        print('Model is already trained.')

In [ ]:
# Create directories to store results
try:
    Path('./models').mkdir()
    print('`./models` directory successfully created.')
except Exception as e:
    print(str(e))
try:
    Path('./metrics').mkdir()
    print('`./metrics` directory successfully created.')
except Exception as e:
    print(str(e))

In [ ]:
# Instantiate our classifier
model_deberta_fine = TweetClassifier(base_model='deberta')
load_model(model_deberta_fine)

Finally, do the training (fine-tuning).

In [ ]:
# Train on train, evaluate on val
train_model(model_deberta_fine, train[['doc', target]], val[['doc', target]], epochs=6, batch_size=64)

In [ ]:
plot_metrics(model_deberta_fine.name, epochs=6)

##### <a id="predict"></a>4. Make predictions

Now, we can predict the tweet's class for the `test` data and submit results.

In [ ]:
labels_pred = []
for doc in tqdm(test['doc']):
    labels_pred.append(model_deberta_fine.model(**model_deberta_fine.tokenizer(doc, return_tensors='pt').to(device)).logits.argmax(-1).item())
df_pred = pd.DataFrame({'id': test['id'], 'target': labels_pred})

In [ ]:
# Submission
df_pred.to_csv('/kaggle/working/submission.csv', index=False)